# 예제 02. 도구 명세와 레지스트리 만들기

전산실험 2 · AI Agent 만들기 · **5주차** · 예상 소요 35분

## 목표
- 도구를 **작게, 하나만 하게** 쪼갠다
- 모델이 읽을 **명세**를 설계한다
- `call()` 이 예외를 던지지 않게 만든다
- **실패도 관찰**로 되돌려 모델이 고치게 한다

### 준비 — 공용 모듈과 데이터 만들기 / Setup — write the shared modules and data

이 수업 저장소에는 노트북(`.ipynb`)만 둡니다. 그래서 실습에 필요한 공용 모듈과
데이터를 **아래 셀들이 직접 파일로 만들어 냅니다.** 내려받지 않으므로 네트워크가
느려도, 저장소가 비공개로 바뀌어도 그대로 동작합니다.

아래 셀을 **위에서부터 차례로 한 번씩** 실행한 뒤 다음으로 넘어가세요.
내용을 지금 읽을 필요는 없습니다. 만들어진 `.py` 파일은 왼쪽 파일 탐색기에서
언제든 열어 볼 수 있습니다.

The course repository contains notebooks only. The cells below **write the shared
modules and data files themselves** instead of downloading them. Run them once,
top to bottom, then continue. You do not need to read them now — the generated
`.py` files can be opened from the file browser on the left.

In [ ]:
%%writefile kmu_llm.py
"""전산실험2 - AI Agent 만들기 : 공용 LLM 래퍼.

계명대 통계학과 서버와 Google Gemini를 같은 방식으로 호출한다.
둘 다 OpenAI 호환 API를 제공하므로 openai SDK 하나로 통일하고
base_url 과 model 만 바꾼다.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 이 저장소는 공개(public)이므로 **학과 서버 주소와 API 키를 코드에 적지
 않는다.** 둘 다 수업시간에 공개하고, 각자 Colab Secrets 에 저장해서 쓴다.

     KMU_BASE_URL  = 수업시간에 공개  (예: https://.../v1)
     KMU_API_KEY   = 수업시간에 공개

 왼쪽 열쇠 아이콘(🔑) → 새 secret → 노트북 액세스 토글 켜기
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

사용 예:
    from kmu_llm import LLM, ask, check_server

    check_server()                       # 서버가 살아 있는지 먼저 확인
    print(ask("표준편차를 한 문장으로 설명해줘"))

    llm = LLM(provider="gemini")         # 백업 경로
    print(llm.chat([{"role": "user", "content": "안녕"}]))
"""

from __future__ import annotations

import os
import time
from typing import Any, Iterator

__all__ = [
    "KMU_MODEL",
    "GEMINI_BASE_URL",
    "GEMINI_MODEL",
    "PROVIDERS",
    "LLM",
    "ask",
    "get_secret",
    "get_key",
    "get_base_url",
    "check_server",
]

# --------------------------------------------------------------------------
# 서버 설정
# --------------------------------------------------------------------------
# 학과 서버 주소는 여기에 적지 않는다. Colab Secrets 의 KMU_BASE_URL 에서 읽는다.
# 모델 태그는 공개해도 무방하므로 상수로 둔다.
KMU_MODEL = os.environ.get("KMU_MODEL", "gemma4")

# 백업 경로. Google AI Studio 에서 무료 키를 발급받아 쓴다. (공개 주소)
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEMINI_MODEL = "gemini-2.0-flash"

PROVIDERS: dict[str, dict[str, Any]] = {
    "kmu": {
        "base_url": None,  # 실행 시점에 Secrets 에서 읽는다
        "base_url_key": "KMU_BASE_URL",
        "model": KMU_MODEL,
        "key_name": "KMU_API_KEY",
        "label": "계명대 학과 서버",
    },
    "gemini": {
        "base_url": GEMINI_BASE_URL,
        "base_url_key": None,
        "model": GEMINI_MODEL,
        "key_name": "GEMINI_API_KEY",
        "label": "Google Gemini (백업)",
    },
}


# --------------------------------------------------------------------------
# 비밀정보 가져오기
# --------------------------------------------------------------------------
def get_secret(name: str, *, prompt: bool = True, secret: bool = True) -> str:
    """설정값을 안전한 순서로 찾는다.

    1) Colab Secrets (왼쪽 열쇠 아이콘)  2) 환경변수  3) 직접 입력

    값을 코드에 적지 않는 것이 핵심이다. 노트북이나 모듈을 GitHub 에 올리는
    순간 코드에 적힌 값은 전 세계에 공개된다. API 키뿐 아니라 **내부 서버
    주소도** 마찬가지다.

    Parameters
    ----------
    secret : True 면 입력을 화면에 표시하지 않는다(getpass).
             주소처럼 오타 확인이 중요한 값은 False 로 준다.
    """
    # 1) Colab Secrets
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass

    # 2) 환경변수
    value = os.environ.get(name)
    if value:
        return value.strip()

    # 3) 직접 입력
    if not prompt:
        raise RuntimeError(
            f"'{name}' 를 찾을 수 없습니다.\n"
            f"  Colab 왼쪽 열쇠 아이콘(🔑) → 새 secret → 이름 '{name}'\n"
            f"  값은 수업시간에 공개합니다. 노트북 액세스 토글도 켜세요."
        )

    print(f"[{name}] 를 찾지 못했습니다. 직접 입력해 주세요.")
    print(f"   (Colab Secrets 에 '{name}' 로 저장해 두면 다시 묻지 않습니다)")

    if secret:
        import getpass

        value = getpass.getpass(f"{name}: ").strip()
    else:
        value = input(f"{name}: ").strip()

    if not value:
        raise RuntimeError(f"'{name}' 에 빈 값이 입력되었습니다.")
    os.environ[name] = value  # 같은 런타임 안에서는 다시 묻지 않는다
    return value


def get_key(key_name: str = "KMU_API_KEY", *, prompt: bool = True) -> str:
    """API 키를 가져온다. (get_secret 의 별칭 — 1주차 이름 유지)"""
    return get_secret(key_name, prompt=prompt, secret=True)


def get_base_url(provider: str = "kmu", *, prompt: bool = True) -> str:
    """서버 주소를 가져온다. 끝의 '/' 는 떼고 돌려준다.

    학과 서버 주소는 저장소에 적지 않으므로 Secrets 에서 읽는다.
    '/chat/completions' 까지 붙여서 저장한 경우도 알아서 잘라낸다.
    """
    conf = PROVIDERS[provider]

    if conf["base_url"]:
        return str(conf["base_url"]).rstrip("/")

    url = get_secret(conf["base_url_key"], prompt=prompt, secret=False).rstrip("/")

    # openai SDK 는 base_url 뒤에 /chat/completions 를 스스로 붙인다.
    # 주소를 통째로 저장한 경우를 대비해 정리해 준다.
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]

    conf["base_url"] = url  # 같은 세션에서는 재사용
    return url


# --------------------------------------------------------------------------
# LLM 클라이언트
# --------------------------------------------------------------------------
class LLM:
    """OpenAI 호환 서버를 감싼 얇은 클라이언트.

    Parameters
    ----------
    provider : "kmu" | "gemini"
        어느 서버를 쓸지. 기본은 학과 서버.
    model, base_url, api_key :
        직접 지정하면 기본값을 덮어쓴다. (보통 지정하지 않는다)
    """

    def __init__(
        self,
        provider: str = "kmu",
        model: str | None = None,
        base_url: str | None = None,
        api_key: str | None = None,
        timeout: float = 120.0,
    ) -> None:
        if provider not in PROVIDERS:
            raise ValueError(
                f"알 수 없는 provider: {provider!r}. 가능한 값: {list(PROVIDERS)}"
            )
        conf = PROVIDERS[provider]

        self.provider = provider
        self.label = conf["label"]
        self.model = model or conf["model"]
        self.base_url = base_url or get_base_url(provider)
        self.api_key = api_key or get_key(conf["key_name"])

        try:
            from openai import OpenAI
        except ImportError as exc:  # pragma: no cover
            raise ImportError(
                "openai 패키지가 필요합니다.  !pip install -q openai"
            ) from exc

        self.client = OpenAI(
            api_key=self.api_key, base_url=self.base_url, timeout=timeout
        )

    def __repr__(self) -> str:
        # 주소는 출력하지 않는다. 화면 공유·스크린샷으로 새어나가기 쉽다.
        return f"LLM(provider={self.provider!r}, model={self.model!r})"

    # -- 기본 호출 --------------------------------------------------------
    def raw(self, messages: list[dict[str, str]], **kw: Any):
        """응답 객체 전체를 그대로 돌려준다. usage(토큰 수) 확인용."""
        params: dict[str, Any] = {
            "model": self.model,
            "messages": messages,
            "temperature": 0.7,
            "max_tokens": 1024,
        }
        params.update(kw)
        return self.client.chat.completions.create(**params)

    def chat(self, messages: list[dict[str, str]], **kw: Any) -> str:
        """대화 메시지를 보내고 응답 '텍스트'만 돌려준다."""
        response = self.raw(messages, **kw)
        content = response.choices[0].message.content
        return (content or "").strip()

    def stream(self, messages: list[dict[str, str]], **kw: Any) -> Iterator[str]:
        """응답을 조각(chunk) 단위로 흘려보낸다. 타이핑되는 효과."""
        kw["stream"] = True
        for chunk in self.raw(messages, **kw):
            if not chunk.choices:
                continue
            piece = chunk.choices[0].delta.content
            if piece:
                yield piece

    # -- 편의 메서드 ------------------------------------------------------
    def ask(self, prompt: str, system: str | None = None, **kw: Any) -> str:
        """한 번짜리 질문. messages 를 매번 만들기 번거로울 때 쓴다."""
        messages: list[dict[str, str]] = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        return self.chat(messages, **kw)

    def ask_retry(
        self,
        prompt: str,
        system: str | None = None,
        *,
        n_retry: int = 3,
        wait: float = 2.0,
        **kw: Any,
    ) -> str:
        """일시적인 네트워크·서버 오류를 재시도한다.

        30회 반복 실험처럼 호출이 많은 실습에서 한 번의 실패로 전체가
        멈추지 않도록 하기 위한 것이다.
        """
        last_error: Exception | None = None
        for attempt in range(1, n_retry + 1):
            try:
                return self.ask(prompt, system, **kw)
            except Exception as exc:  # 네트워크·서버 오류 전반
                last_error = exc
                if attempt < n_retry:
                    print(f"   [재시도 {attempt}/{n_retry - 1}] {type(exc).__name__}")
                    time.sleep(wait * attempt)  # 점점 더 오래 기다린다
        raise RuntimeError(f"{n_retry}회 시도 모두 실패했습니다.") from last_error

    def count_tokens(self, messages: list[dict[str, str]]) -> int:
        """입력 토큰 수를 서버에 물어본다. 출력은 1토큰으로 잘라 낭비를 줄인다."""
        return self.raw(messages, max_tokens=1).usage.prompt_tokens


# --------------------------------------------------------------------------
# 모듈 수준 편의 함수
# --------------------------------------------------------------------------
_default_llm: LLM | None = None


def ask(
    prompt: str, system: str | None = None, *, provider: str = "kmu", **kw: Any
) -> str:
    """가장 간단한 한 줄 호출.  ask("안녕") -> "안녕하세요..."

    내부적으로 LLM 객체를 한 번만 만들어 재사용한다.
    """
    global _default_llm
    if _default_llm is None or _default_llm.provider != provider:
        _default_llm = LLM(provider=provider)
    return _default_llm.ask(prompt, system, **kw)


# --------------------------------------------------------------------------
# 서버 진단
# --------------------------------------------------------------------------
def check_server(provider: str = "kmu", base_url: str | None = None) -> dict[str, Any]:
    """서버가 살아 있는지, 어떤 모델이 올라가 있는지 확인한다.

    수업 첫 실습. 실패하면 무엇을 확인해야 하는지 함께 출력한다.

    Returns
    -------
    dict : {"ok": bool, "models": list[str], "error": str | None}
    """
    import requests

    conf = PROVIDERS[provider]
    base_url = (base_url or get_base_url(provider)).rstrip("/")
    api_key = get_key(conf["key_name"])

    # 주소 전체를 출력하지 않는다. 호스트만 가려서 보여준다.
    print(f"조회 중: GET {_mask_url(base_url)}/models")

    try:
        response = requests.get(
            f"{base_url}/models",
            headers={"Authorization": f"Bearer {api_key}"},
            timeout=15,
        )
    except Exception as exc:
        print(f"\n[실패] 서버에 연결하지 못했습니다: {type(exc).__name__}")
        _print_troubleshooting()
        return {"ok": False, "models": [], "error": str(exc)}

    if response.status_code != 200:
        print(f"\n[실패] HTTP {response.status_code}")
        print(f"   응답: {response.text[:300]}")
        if response.status_code in (401, 403):
            print("   -> API 키가 잘못되었거나 만료되었을 가능성이 높습니다.")
        elif response.status_code == 404:
            print("   -> KMU_BASE_URL 이 '/v1' 로 끝나는지 확인하세요.")
        return {"ok": False, "models": [], "error": f"HTTP {response.status_code}"}

    models = [m.get("id", "?") for m in response.json().get("data", [])]

    print(f"\n[성공] 서버가 응답했습니다. 사용 가능한 모델 {len(models)}개:")
    for name in models:
        mark = "  <- 기본 모델" if name == conf["model"] else ""
        print(f"   - {name}{mark}")

    if conf["model"] not in models and models:
        print(
            f"\n[주의] 설정된 모델 '{conf['model']}' 이 목록에 없습니다.\n"
            f"       위 목록에서 골라 LLM(model='...') 로 지정하거나\n"
            f"       환경변수 KMU_MODEL 을 설정하세요."
        )

    return {"ok": True, "models": models, "error": None}


def _mask_url(url: str) -> str:
    """화면 공유·스크린샷 유출을 막기 위해 호스트를 가린다."""
    import re

    return re.sub(r"//([^/]+)", lambda m: "//" + m.group(1)[:4] + "***", url, count=1)


def _print_troubleshooting() -> None:
    """연결 실패 시 확인할 것들."""
    print("\n확인해 볼 것:")
    print("   1. Colab Secrets 의 KMU_BASE_URL 값이 정확합니까?")
    print("      - http / https 를 맞게 적었습니까?")
    print("      - 포트가 필요할 수 있습니다.")
    print("      - 끝이 '/v1' 이어야 합니다. ('/chat/completions' 는 빼도 됩니다)")
    print("   2. 학과 서버가 켜져 있습니까? 교내망에서만 열려 있지 않습니까?")
    print("   3. 다른 주소를 시험해 보려면:")
    print("      check_server(base_url='...')")
    print("   4. 계속 안 되면 백업 경로를 쓰세요:  LLM(provider='gemini')")


In [ ]:
%%writefile tools.py
"""전산실험2 5주차 : 도구(Tool)와 도구 레지스트리.

에이전트가 바깥 세계에 손을 뻗는 방법.

핵심 원칙:
    **계산은 모델이 아니라 도구가 한다.**

    LLM은 "어떤 도구를 어떤 인자로 부를지" 만 정한다.
    평균을 구하고 p값을 계산하는 것은 numpy 와 scipy 가 한다.
    1주차에 확인했듯 LLM은 난수도 제대로 못 뽑는다. 계산을 맡기면 안 된다.

구성:
    Tool          이름 · 설명 · 인자 명세 · 실제 함수
    ToolResult    실행 결과 (성공/실패, 값, 모델에게 돌려줄 관찰 문자열)
    ToolRegistry  도구를 등록·직렬화·호출한다
    StatToolbox   DataFrame 에 묶인 통계 도구 모음

사용 예:
    import pandas as pd
    from tools import StatToolbox

    box = StatToolbox(pd.read_csv("sample_clinic.csv"))
    print(box.describe())                     # 프롬프트에 넣을 도구 목록
    r = box.call("compare_two_groups",
                 {"value_column": "bmi", "group_column": "group"})
    print(r.observation())
"""

from __future__ import annotations

import json
from dataclasses import dataclass, field
from typing import Any, Callable

__all__ = ["Tool", "ToolResult", "ToolRegistry", "StatToolbox"]


# --------------------------------------------------------------------------
# 도구 정의
# --------------------------------------------------------------------------
@dataclass
class Tool:
    """도구 하나.

    `parameters` 형식::

        {
            "column": {"type": "str", "desc": "분석할 열 이름", "required": True},
            "alpha":  {"type": "float", "desc": "유의수준", "default": 0.05},
            "method": {"type": "str", "enum": ["pearson", "spearman"]},
        }
    """

    name: str
    description: str
    parameters: dict[str, dict[str, Any]]
    func: Callable[..., Any]

    def spec_line(self) -> str:
        """프롬프트에 넣을 한 줄 명세."""
        args = []
        for pname, p in self.parameters.items():
            t = p.get("type", "str")
            mark = "" if p.get("required") else "?"
            enum = f"={'|'.join(map(str, p['enum']))}" if "enum" in p else ""
            args.append(f"{pname}{mark}: {t}{enum}")
        return f"- {self.name}({', '.join(args)}) — {self.description}"

    def validate(self, args: dict[str, Any]) -> list[str]:
        """인자를 검사한다. 문제 목록을 돌려준다 (빈 목록 = 통과).

        3주차 `structured.validate()` 와 같은 발상이다.
        **모델이 준 것을 믿지 않고 코드가 확인한다.**
        """
        problems: list[str] = []

        for pname, p in self.parameters.items():
            if p.get("required") and pname not in args:
                problems.append(f"필수 인자 '{pname}' 가 없습니다.")

        known = set(self.parameters)
        for given in args:
            if given not in known:
                problems.append(
                    f"'{given}' 는 이 도구의 인자가 아닙니다. 가능한 인자: {sorted(known)}"
                )

        for pname, value in args.items():
            p = self.parameters.get(pname)
            if p is None:
                continue
            if "enum" in p and value not in p["enum"]:
                problems.append(
                    f"'{pname}' 의 값 {value!r} 는 허용되지 않습니다. 가능한 값: {p['enum']}"
                )
            expected = p.get("type")
            if expected == "int" and not isinstance(value, int):
                problems.append(f"'{pname}' 는 정수여야 합니다 (받은 값: {value!r}).")
            elif expected == "float" and not isinstance(value, (int, float)):
                problems.append(f"'{pname}' 는 숫자여야 합니다 (받은 값: {value!r}).")
            elif expected == "bool" and not isinstance(value, bool):
                problems.append(f"'{pname}' 는 true/false 여야 합니다 (받은 값: {value!r}).")
            elif expected == "str" and not isinstance(value, str):
                problems.append(f"'{pname}' 는 문자열이어야 합니다 (받은 값: {value!r}).")
        return problems


@dataclass
class ToolResult:
    """도구 실행 결과."""

    tool: str
    ok: bool
    value: Any = None
    error: str | None = None
    hint: str | None = None      # 실패했을 때 다음에 무엇을 하면 되는지

    def observation(self, max_chars: int = 1200) -> str:
        """모델에게 돌려줄 문자열.

        **실패도 관찰이다.** 조용히 넘기지 않고 무엇이 왜 안 됐는지 알려준다.
        그래야 모델이 다음 행동을 고칠 수 있다 (3주차 되먹임과 같은 구조).
        """
        if not self.ok:
            msg = f"[{self.tool} 실패] {self.error}"
            if self.hint:
                msg += f"\n힌트: {self.hint}"
            return msg

        if isinstance(self.value, str):
            body = self.value
        else:
            body = json.dumps(self.value, ensure_ascii=False, indent=2, default=str)
        if len(body) > max_chars:
            body = body[:max_chars] + f"\n... (생략, 전체 {len(body)}자)"
        return f"[{self.tool} 결과]\n{body}"


# --------------------------------------------------------------------------
# 레지스트리
# --------------------------------------------------------------------------
class ToolRegistry:
    """도구를 등록하고, 프롬프트용으로 직렬화하고, 호출한다."""

    def __init__(self) -> None:
        self._tools: dict[str, Tool] = {}
        self.log: list[ToolResult] = []      # 실행 이력 (6주차 trace 의 씨앗)

    # -- 등록 -------------------------------------------------------------
    def register(self, tool: Tool) -> Tool:
        self._tools[tool.name] = tool
        return tool

    def tool(self, name: str, description: str, parameters: dict | None = None):
        """데코레이터로 등록한다.

            @registry.tool("mean", "평균을 구한다", {"col": {"type": "str", "required": True}})
            def _(col): ...
        """
        def deco(func: Callable[..., Any]) -> Callable[..., Any]:
            self.register(Tool(name, description, parameters or {}, func))
            return func

        return deco

    # -- 조회 -------------------------------------------------------------
    def names(self) -> list[str]:
        return sorted(self._tools)

    def __contains__(self, name: object) -> bool:
        return name in self._tools

    def __getitem__(self, name: str) -> Tool:
        return self._tools[name]

    def __len__(self) -> int:
        return len(self._tools)

    def describe(self) -> str:
        """프롬프트에 그대로 넣을 도구 목록."""
        lines = [t.spec_line() for _, t in sorted(self._tools.items())]
        return "\n".join(lines)

    # -- 실행 -------------------------------------------------------------
    def call(self, name: str, args: dict[str, Any] | None = None) -> ToolResult:
        """도구를 실행한다. **예외를 밖으로 던지지 않는다.**

        에이전트 루프 안에서 도구가 예외를 던지면 루프 전체가 죽는다.
        실패를 `ToolResult` 로 감싸 돌려주면 모델이 보고 고칠 수 있다.
        """
        args = args or {}

        if name not in self._tools:
            result = ToolResult(
                name, False,
                error=f"'{name}' 라는 도구가 없습니다.",
                hint=f"사용 가능한 도구: {self.names()}",
            )
            self.log.append(result)
            return result

        tool = self._tools[name]
        problems = tool.validate(args)
        if problems:
            result = ToolResult(
                name, False,
                error="인자가 잘못되었습니다: " + "; ".join(problems),
                hint=tool.spec_line(),
            )
            self.log.append(result)
            return result

        try:
            value = tool.func(**args)
            result = ToolResult(name, True, value=value)
        except Exception as exc:
            result = ToolResult(
                name, False,
                error=f"{type(exc).__name__}: {exc}",
                hint="인자 값을 확인하세요. 열 이름은 list_columns 로 확인할 수 있습니다.",
            )
        self.log.append(result)
        return result


# --------------------------------------------------------------------------
# 통계 도구 모음
# --------------------------------------------------------------------------
class StatToolbox(ToolRegistry):
    """DataFrame 하나에 묶인 통계 도구들.

    도구는 **작고, 하나만 하고, 결과를 구조화해서** 돌려줘야 한다.
    거대한 만능 도구를 하나 만들면 모델이 인자를 제대로 채우지 못한다.
    """

    def __init__(self, df: Any, name: str = "data") -> None:
        super().__init__()
        self.df = df
        self.name = name
        self._register_all()

    # -- 내부 헬퍼 --------------------------------------------------------
    def _numeric(self, column: str):
        """열을 수치로 강제 변환하고 결측을 뺀다.

        실제 데이터에는 '58세', ' 47 ' 같은 값이 섞여 있다.
        도구가 이것을 조용히 통과시키면 이후 계산이 전부 틀어진다.
        """
        import pandas as pd

        if column not in self.df.columns:
            raise KeyError(f"'{column}' 열이 없습니다. 있는 열: {list(self.df.columns)}")
        s = pd.to_numeric(self.df[column], errors="coerce")
        n_bad = int(s.isna().sum() - self.df[column].isna().sum())
        s = s.dropna()
        if len(s) < 3:
            raise ValueError(f"'{column}' 의 유효한 수치가 {len(s)}개뿐입니다.")
        return s, n_bad

    def _check_columns(self, *columns: str) -> None:
        missing = [c for c in columns if c not in self.df.columns]
        if missing:
            raise KeyError(f"{missing} 열이 없습니다. 있는 열: {list(self.df.columns)}")

    # -- 도구 등록 --------------------------------------------------------
    def _register_all(self) -> None:
        import numpy as np
        import pandas as pd
        from scipy import stats

        # ---------------------------------------------------------------
        @self.tool(
            "list_columns",
            "데이터의 열 이름, 타입, 결측 수, 고유값 수를 돌려준다. 항상 먼저 호출할 것.",
            {},
        )
        def _list_columns() -> dict:
            rows = []
            for c in self.df.columns:
                s = self.df[c]
                num = pd.to_numeric(s, errors="coerce")
                rows.append({
                    "column": c,
                    "dtype": str(s.dtype),
                    "n_missing": int(s.isna().sum()),
                    "n_unique": int(s.nunique(dropna=True)),
                    "numeric_ok": bool(num.notna().sum() > len(s) * 0.8),
                    "sample": [str(v) for v in s.dropna().head(3).tolist()],
                })
            return {"n_rows": int(len(self.df)), "columns": rows}

        # ---------------------------------------------------------------
        @self.tool(
            "describe_column",
            "한 열의 기술통계를 돌려준다. 수치형이면 평균·표준편차·사분위수, 범주형이면 빈도표.",
            {"column": {"type": "str", "desc": "열 이름", "required": True}},
        )
        def _describe_column(column: str) -> dict:
            self._check_columns(column)
            s = self.df[column]
            num = pd.to_numeric(s, errors="coerce")

            if num.notna().sum() > len(s) * 0.8:      # 수치형으로 본다
                v = num.dropna()
                q1, q3 = v.quantile([0.25, 0.75])
                return {
                    "column": column, "kind": "numeric", "n": int(len(v)),
                    "n_missing": int(s.isna().sum()),
                    "n_non_numeric": int(num.isna().sum() - s.isna().sum()),
                    "mean": round(float(v.mean()), 4),
                    "sd": round(float(v.std(ddof=1)), 4),
                    "min": round(float(v.min()), 4),
                    "q1": round(float(q1), 4),
                    "median": round(float(v.median()), 4),
                    "q3": round(float(q3), 4),
                    "max": round(float(v.max()), 4),
                    "skew": round(float(v.skew()), 3),
                }
            counts = s.value_counts(dropna=True)
            return {
                "column": column, "kind": "categorical",
                "n": int(s.notna().sum()), "n_missing": int(s.isna().sum()),
                "n_levels": int(counts.size),
                "counts": {str(k): int(v) for k, v in counts.head(15).items()},
            }

        # ---------------------------------------------------------------
        @self.tool(
            "check_normality",
            "한 열의 정규성을 Shapiro-Wilk 로 검정한다. 검정 방법을 고르기 전에 확인할 것.",
            {"column": {"type": "str", "desc": "수치형 열 이름", "required": True},
             "group_column": {"type": "str", "desc": "집단별로 나눠 검정할 열 (선택)"}},
        )
        def _check_normality(column: str, group_column: str | None = None) -> dict:
            if group_column is None:
                v, n_bad = self._numeric(column)
                W, p = stats.shapiro(v[:5000])
                return {"column": column, "n": int(len(v)),
                        "n_non_numeric": n_bad,
                        "statistic_W": round(float(W), 4), "p_value": float(p),
                        "normal_at_0.05": bool(p >= 0.05)}

            self._check_columns(column, group_column)
            out = {}
            work = self.df[[column, group_column]].copy()
            work[column] = pd.to_numeric(work[column], errors="coerce")
            for g, sub in work.dropna().groupby(group_column):
                if len(sub) < 3:
                    out[str(g)] = {"n": int(len(sub)), "error": "표본이 3 미만"}
                    continue
                W, p = stats.shapiro(sub[column][:5000])
                out[str(g)] = {"n": int(len(sub)),
                               "statistic_W": round(float(W), 4),
                               "p_value": float(p),
                               "normal_at_0.05": bool(p >= 0.05)}
            return {"column": column, "by": group_column, "groups": out}

        # ---------------------------------------------------------------
        @self.tool(
            "compare_two_groups",
            "두 독립 집단의 평균(또는 분포)을 비교한다. 등분산은 Welch 로 보정하며, "
            "method=auto 면 정규성 검정 결과에 따라 t검정/Mann-Whitney 를 고른다.",
            {"value_column": {"type": "str", "desc": "비교할 수치형 열", "required": True},
             "group_column": {"type": "str", "desc": "두 수준을 가진 집단 열", "required": True},
             "method": {"type": "str", "enum": ["auto", "t", "mannwhitney"],
                        "desc": "기본 auto"},
             "alpha": {"type": "float", "desc": "유의수준, 기본 0.05"}},
        )
        def _compare_two_groups(value_column: str, group_column: str,
                                method: str = "auto", alpha: float = 0.05) -> dict:
            self._check_columns(value_column, group_column)
            work = self.df[[value_column, group_column]].copy()
            work[value_column] = pd.to_numeric(work[value_column], errors="coerce")
            work = work.dropna()

            levels = sorted(work[group_column].astype(str).unique())
            if len(levels) != 2:
                raise ValueError(
                    f"'{group_column}' 의 수준이 {len(levels)}개입니다 ({levels[:5]}). "
                    f"두 집단 비교에는 2개여야 합니다. 3개 이상이면 compare_many_groups 를 쓰세요."
                )
            a = work[work[group_column].astype(str) == levels[0]][value_column]
            b = work[work[group_column].astype(str) == levels[1]][value_column]

            chosen = method
            if method == "auto":
                pa = stats.shapiro(a[:5000]).pvalue if len(a) >= 3 else 0.0
                pb = stats.shapiro(b[:5000]).pvalue if len(b) >= 3 else 0.0
                chosen = "t" if (pa >= 0.05 and pb >= 0.05) else "mannwhitney"

            if chosen == "t":
                res = stats.ttest_ind(a, b, equal_var=False)      # Welch
                test_name = "Welch independent t-test"
                # Hedges 보정 없는 Cohen's d (pooled sd)
                sp = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1))
                             / (len(a) + len(b) - 2))
                effect = {"name": "Cohen's d", "value": round(float((a.mean() - b.mean()) / sp), 4)}
            else:
                res = stats.mannwhitneyu(a, b, alternative="two-sided")
                test_name = "Mann-Whitney U"
                # rank-biserial correlation
                r = 1 - (2 * res.statistic) / (len(a) * len(b))
                effect = {"name": "rank-biserial r", "value": round(float(r), 4)}

            return {
                "test": test_name,
                "method_requested": method, "method_used": chosen,
                "groups": {levels[0]: {"n": int(len(a)), "mean": round(float(a.mean()), 4),
                                       "sd": round(float(a.std(ddof=1)), 4),
                                       "median": round(float(a.median()), 4)},
                           levels[1]: {"n": int(len(b)), "mean": round(float(b.mean()), 4),
                                       "sd": round(float(b.std(ddof=1)), 4),
                                       "median": round(float(b.median()), 4)}},
                "statistic": round(float(res.statistic), 4),
                "p_value": float(res.pvalue),
                "alpha": alpha,
                "significant": bool(res.pvalue < alpha),
                "effect_size": effect,
            }

        # ---------------------------------------------------------------
        @self.tool(
            "compare_paired",
            "같은 대상에서 두 번 측정한 값(전/후)을 비교한다. 대응표본 t검정 또는 Wilcoxon.",
            {"before_column": {"type": "str", "desc": "이전 측정값 열", "required": True},
             "after_column": {"type": "str", "desc": "이후 측정값 열", "required": True},
             "method": {"type": "str", "enum": ["auto", "t", "wilcoxon"]},
             "alpha": {"type": "float"}},
        )
        def _compare_paired(before_column: str, after_column: str,
                            method: str = "auto", alpha: float = 0.05) -> dict:
            self._check_columns(before_column, after_column)
            work = self.df[[before_column, after_column]].apply(
                lambda c: pd.to_numeric(c, errors="coerce")).dropna()
            if len(work) < 3:
                raise ValueError(f"두 열이 모두 있는 행이 {len(work)}개뿐입니다.")
            before, after = work[before_column], work[after_column]
            diff = after - before

            chosen = method
            if method == "auto":
                chosen = "t" if stats.shapiro(diff[:5000]).pvalue >= 0.05 else "wilcoxon"

            if chosen == "t":
                res = stats.ttest_rel(before, after)
                test_name = "paired t-test"
                effect = {"name": "Cohen's dz",
                          "value": round(float(diff.mean() / diff.std(ddof=1)), 4)}
            else:
                res = stats.wilcoxon(before, after)
                test_name = "Wilcoxon signed-rank"
                effect = {"name": "median difference", "value": round(float(diff.median()), 4)}

            return {
                "test": test_name, "method_used": chosen,
                "n_pairs": int(len(work)),
                "mean_before": round(float(before.mean()), 4),
                "mean_after": round(float(after.mean()), 4),
                "mean_difference": round(float(diff.mean()), 4),
                "statistic": round(float(res.statistic), 4),
                "p_value": float(res.pvalue),
                "alpha": alpha, "significant": bool(res.pvalue < alpha),
                "effect_size": effect,
            }

        # ---------------------------------------------------------------
        @self.tool(
            "compare_many_groups",
            "세 집단 이상의 평균(또는 분포)을 비교한다. 일원분산분석 또는 Kruskal-Wallis.",
            {"value_column": {"type": "str", "required": True},
             "group_column": {"type": "str", "required": True},
             "method": {"type": "str", "enum": ["auto", "anova", "kruskal"]},
             "alpha": {"type": "float"}},
        )
        def _compare_many_groups(value_column: str, group_column: str,
                                 method: str = "auto", alpha: float = 0.05) -> dict:
            self._check_columns(value_column, group_column)
            work = self.df[[value_column, group_column]].copy()
            work[value_column] = pd.to_numeric(work[value_column], errors="coerce")
            work = work.dropna()

            groups = [g[value_column].values for _, g in work.groupby(group_column)]
            labels = [str(k) for k, _ in work.groupby(group_column)]
            if len(groups) < 3:
                raise ValueError(
                    f"집단이 {len(groups)}개입니다. 3개 이상이어야 합니다. "
                    f"2개면 compare_two_groups 를 쓰세요."
                )

            chosen = method
            if method == "auto":
                정규 = all(len(g) >= 3 and stats.shapiro(g[:5000]).pvalue >= 0.05 for g in groups)
                등분산 = stats.levene(*groups).pvalue >= 0.05
                chosen = "anova" if (정규 and 등분산) else "kruskal"

            if chosen == "anova":
                res = stats.f_oneway(*groups)
                test_name = "one-way ANOVA"
                grand = np.concatenate(groups)
                ss_b = sum(len(g) * (g.mean() - grand.mean()) ** 2 for g in groups)
                ss_t = ((grand - grand.mean()) ** 2).sum()
                effect = {"name": "eta squared", "value": round(float(ss_b / ss_t), 4)}
            else:
                res = stats.kruskal(*groups)
                test_name = "Kruskal-Wallis"
                n = len(np.concatenate(groups))
                effect = {"name": "epsilon squared",
                          "value": round(float(res.statistic / ((n ** 2 - 1) / (n + 1))), 4)}

            return {
                "test": test_name, "method_used": chosen,
                "groups": {lab: {"n": int(len(g)), "mean": round(float(g.mean()), 4),
                                 "sd": round(float(g.std(ddof=1)), 4)}
                           for lab, g in zip(labels, groups)},
                "statistic": round(float(res.statistic), 4),
                "p_value": float(res.pvalue),
                "alpha": alpha, "significant": bool(res.pvalue < alpha),
                "effect_size": effect,
                "note": "유의하면 사후검정(Tukey 등)이 필요합니다.",
            }

        # ---------------------------------------------------------------
        @self.tool(
            "test_association",
            "두 범주형 변수의 연관성을 카이제곱 검정한다. 기대빈도가 5 미만이면 Fisher 를 권한다.",
            {"column_a": {"type": "str", "required": True},
             "column_b": {"type": "str", "required": True},
             "alpha": {"type": "float"}},
        )
        def _test_association(column_a: str, column_b: str, alpha: float = 0.05) -> dict:
            self._check_columns(column_a, column_b)
            table = pd.crosstab(self.df[column_a], self.df[column_b])
            if table.size < 4:
                raise ValueError("분할표가 2x2 보다 작습니다.")
            chi2, p, dof, expected = stats.chi2_contingency(table)
            min_exp = float(expected.min())
            n = int(table.values.sum())
            cramers_v = float(np.sqrt(chi2 / (n * (min(table.shape) - 1))))
            return {
                "test": "chi-square test of independence",
                "table": table.to_dict(),
                "chi2": round(float(chi2), 4), "df": int(dof),
                "p_value": float(p), "alpha": alpha,
                "significant": bool(p < alpha),
                "min_expected_count": round(min_exp, 2),
                "effect_size": {"name": "Cramer's V", "value": round(cramers_v, 4)},
                "warning": (None if min_exp >= 5 else
                            "기대빈도가 5 미만인 칸이 있습니다. Fisher 정확검정을 고려하세요."),
            }

        # ---------------------------------------------------------------
        @self.tool(
            "correlate",
            "두 수치형 변수의 상관을 구한다. method=auto 면 정규성에 따라 pearson/spearman 선택.",
            {"column_a": {"type": "str", "required": True},
             "column_b": {"type": "str", "required": True},
             "method": {"type": "str", "enum": ["auto", "pearson", "spearman"]}},
        )
        def _correlate(column_a: str, column_b: str, method: str = "auto") -> dict:
            self._check_columns(column_a, column_b)
            work = self.df[[column_a, column_b]].apply(
                lambda c: pd.to_numeric(c, errors="coerce")).dropna()
            if len(work) < 3:
                raise ValueError(f"두 열이 모두 있는 행이 {len(work)}개뿐입니다.")
            a, b = work[column_a], work[column_b]

            chosen = method
            if method == "auto":
                정규 = (stats.shapiro(a[:5000]).pvalue >= 0.05
                        and stats.shapiro(b[:5000]).pvalue >= 0.05)
                chosen = "pearson" if 정규 else "spearman"

            fn = stats.pearsonr if chosen == "pearson" else stats.spearmanr
            res = fn(a, b)
            return {
                "method_used": chosen, "n": int(len(work)),
                "coefficient": round(float(res[0]), 4),
                "p_value": float(res[1]),
                "note": "상관은 인과가 아닙니다.",
            }


In [ ]:
!pip install -q openai

# 이 수업 저장소에서 공용 모듈 내려받기

from kmu_llm import LLM, ask, check_server

PROVIDER = "kmu"          # 학과 서버가 안 되면 "gemini" 로 바꾸세요
llm = LLM(provider=PROVIDER)
print("준비 완료:", llm)

In [ ]:
%%writefile sample_clinic.csv
patient_id,age,sex,group,bp_before,bp_after,bmi,smoker,satisfaction,income,region
P0001,40.0,M,control,153.9,156.7,25.1,Y,3,12756.0,Busan
P0002,39.0,F,control,146.9,154.0,24.9,N,3,21248.0,Daegu
P0003,49.0,M,treatment,149.4,121.3,31.5,N,4,24254.0,Daegu
P0004,69.0,F,treatment,161.9,155.9,25.7,N,3,56575.0,Daegu
P0005,64.0,M,treatment,168.8,155.2,23.3,N,3,72272.0,Seoul
P0006,48.0,F,treatment,159.8,142.8,25.8,N,4,38254.0,Daegu
P0007,48.0,M,treatment,167.4,156.9,26.9,N,3,7703.0,Daegu
P0008,67.0,M,treatment,480.0,133.9,22.2,N,2,22419.0,Gwangju
P0009,41.0,M,treatment,134.8,111.0,29.2,N,4,22793.0,Seoul
P0010,63.0,F,control,91.7,82.4,24.5,N,2,44750.0,Daegu
P0011,56.0,F,control,143.2,140.8,27.8,N,5,37430.0,Seoul
P0012,53.0,M,control,133.5,129.2,23.7,Y,3,33852.0,Seoul
P0013,61.0,M,treatment,121.7,109.7,29.1,N,4,41307.0,Daegu
P0014,37.0,F,treatment,160.1,143.6,24.4,N,1,53924.0,Daegu
P0015,48.0,F,control,155.3,161.4,30.8,Y,1,92024.0,Gwangju
P0016,30.0,M,control,141.8,137.4,25.4,Y,3,20452.0,Daegu
P0017,70.0,F,control,112.4,111.2,25.2,N,4,23648.0,Seoul
P0018,57.0,M,control,137.3,135.3,20.6,Y,2,10089.0,Seoul
P0019,46.0,F,treatment,123.7,125.5,24.5,N,2,22951.0,Busan
P0020,73.0,F,treatment,162.2,,23.7,Y,3,39878.0,Busan
P0021,57.0,M,control,144.5,134.2,29.9,N,4,75988.0,Busan
P0022,63.0,F,treatment,164.4,157.8,26.0,N,4,65927.0,Busan
P0023,33.0,M,treatment,115.2,109.3,21.5,N,3,40416.0,Busan
P0024,48.0,M,treatment,125.3,108.5,30.6,N,3,19877.0,Gwangju
P0025,44.0,F,control,156.0,161.7,22.5,N,4,44856.0,Seoul
P0026,68.0,M,treatment,156.3,155.5,23.2,Y,1,26890.0,Seoul
P0027,63.0,M,treatment,135.4,111.0,,N,3,7287.0,Daegu
P0028,59.0,F,treatment,132.9,117.3,,N,2,16354.0,Daegu
P0029,43.0,M,control,152.7,156.8,23.0,N,4,26488.0,Daegu
P0030,69.0,F,control,144.4,133.7,24.8,N,2,10399.0,Gwangju
P0031,35.0,F,control,138.2,132.3,27.7,Y,4,10883.0,Busan
P0032,43.0,F,treatment,135.8,117.1,31.1,N,1,36864.0,Busan
P0033,50.0,F,control,156.0,143.6,28.7,N,1,24430.0,Seoul
P0034,50.0,F,treatment,158.9,146.9,23.8,N,3,22788.0,Daegu
P0035,65.0,F,control,162.9,156.7,29.6,N,1,10734.0,Daegu
P0036,54.0,F,treatment,162.2,143.5,27.4,N,2,136149.0,Busan
P0037,69.0,F,control,133.8,127.7,28.6,N,2,50707.0,Seoul
P0038,37.0,M,control,142.0,149.3,23.9,Y,3,18634.0,Daegu
P0039,52.0,M,treatment,165.0,148.7,26.0,N,1,54997.0,Gwangju
P0040,68.0,F,treatment,132.9,114.7,23.6,N,2,17947.0,Daegu
P0041,41.0,F,control,152.3,146.6,23.3,N,1,42218.0,Seoul
P0042,48.0,F,treatment,152.7,153.0,28.3,N,3,11540.0,Daegu
P0043,58세,F,control,138.4,142.3,27.9,N,1,33251.0,Seoul
P0044,68.0,F,treatment,149.0,122.4,32.2,N,1,57593.0,Daegu
P0045,63.0,F,treatment,160.6,152.0,24.2,N,5,36447.0,Busan
P0046,35.0,M,control,172.7,177.5,28.0,Y,4,12268.0,Gwangju
P0047,45.0,F,treatment,141.6,121.4,21.0,Y,5,100463.0,Daegu
P0048,64.0,F,treatment,142.5,117.9,24.1,N,5,21024.0,Daegu
P0049,62.0,M,treatment,151.6,136.5,29.1,N,4,16482.0,Seoul
P0050,88.0,M,control,126.8,112.0,32.0,Y,2,46835.0,Seoul
P0051,62.0,F,control,147.8,146.1,30.9,Y,5,19540.0,Seoul
P0052,51.0,F,treatment,127.1,105.1,28.3,Y,5,60886.0,Daegu
P0053,61.0,F,control,156.6,,31.1,Y,3,22335.0,Seoul
P0054,63.0,M,control,142.9,137.9,23.4,N,4,21115.0,Gwangju
P0055,44.0,F,control,163.1,165.8,26.2,N,1,17212.0,Daegu
P0056,50.0,F,control,134.5,131.7,30.4,N,2,48016.0,Daegu
P0057,46.0,F,treatment,148.8,134.7,26.7,N,2,5933.0,Seoul
P0058,43.0,M,treatment,149.4,127.6,25.2,N,3,55878.0,Busan
P0059,69.0,F,control,145.2,136.6,24.0,N,5,22919.0,Seoul
P0060,39.0,F,control,159.1,153.1,30.4,Y,3,38223.0,Daegu
P0061,52.0,M,treatment,135.1,135.7,27.0,N,3,28871.0,Gwangju
P0062,53.0,M,control,150.1,147.0,16.5,N,5,15375.0,Daegu
P0063,37.0,M,control,144.8,136.7,23.1,N,1,17474.0,Gwangju
P0064,55.0,M,treatment,137.7,126.9,29.0,Y,4,15744.0,Busan
P0065,57.0,F,treatment,166.2,153.8,20.1,N,4,25969.0,Gwangju
P0066,55.0,F,treatment,145.1,137.3,24.0,Y,3,41480.0,Busan
P0067,51.0,F,control,140.6,136.7,25.2,N,4,9821.0,Busan
P0068,68.0,F,treatment,161.3,,24.2,N,2,21397.0,Gwangju
P0069,58.0,M,control,143.2,139.3,16.0,N,1,41523.0,Busan
P0070,86.0,F,control,145.2,146.3,27.0,Y,5,47069.0,Seoul
P0071,77.0,F,control,136.6,150.1,30.5,Y,5,11059.0,Seoul
P0072,69.0,M,control,157.1,156.2,24.5,Y,4,27664.0,Gwangju
P0073,47.0,M,control,156.2,155.2,20.7,Y,5,43633.0,Daegu
P0074,43.0,F,control,133.9,132.4,24.6,Y,4,28681.0,Daegu
P0075,62.0,F,treatment,132.1,125.4,32.7,N,2,24491.0,Busan
P0076,73.0,F,control,152.9,152.8,25.9,N,1,17686.0,Daegu
P0077,55.0,M,control,155.6,143.1,21.2,Y,4,7742.0,Daegu
P0078,51.0,M,treatment,140.6,127.6,26.8,N,3,34283.0,Seoul
P0079,52.0,F,treatment,153.8,141.9,25.0,Y,5,8775.0,Daegu
P0080,51.0,M,control,163.9,145.5,,Y,2,37663.0,Daegu
P0081,52.0,M,treatment,137.3,120.4,22.6,Y,2,22161.0,Seoul
P0082,41.0,F,control,159.0,145.6,20.8,N,4,54669.0,Daegu
P0083,53.0,M,control,135.9,137.9,30.1,N,1,30137.0,Seoul
P0084,49.0,M,control,152.6,150.7,32.6,Y,4,43018.0,Seoul
P0085,48.0,F,control,145.7,149.6,28.8,N,3,42605.0,Daegu
P0086,46.0,F,control,152.8,152.3,30.5,N,4,20892.0,Busan
P0087,69.0,M,treatment,168.0,146.2,28.4,Y,4,39125.0,Daegu
P0088,68.0,F,control,134.1,135.3,24.6,N,5,25873.0,Seoul
P0089,42.0,M,control,126.5,120.9,23.6,N,1,27999.0,Daegu
P0090,50.0,F,treatment,153.2,144.1,28.9,N,5,38156.0,Gwangju
P0091,53.0,M,control,167.5,157.8,24.4,Y,5,18802.0,Daegu
P0092, 47 ,F,control,143.4,139.1,30.8,N,2,49663.0,Seoul
P0093,55.0,M,control,127.7,123.9,20.6,Y,5,71117.0,Daegu
P0094,64.0,F,control,115.5,113.1,27.1,N,4,18509.0,Gwangju
P0095,69.0,M,control,156.0,159.8,26.6,N,3,44618.0,Daegu
P0096,50.0,F,treatment,165.2,153.3,25.9,Y,2,21331.0,Daegu
P0097,46.0,F,treatment,126.7,,25.2,N,4,26323.0,Busan
P0098,57.0,F,control,133.5,133.3,26.9,N,5,36500.0,Daegu
P0099,51.0,F,control,147.6,148.3,21.9,N,1,21461.0,Seoul
P0100,53.0,F,treatment,143.4,132.1,24.7,N,4,30403.0,Gwangju
P0101,71.0,F,treatment,157.0,153.9,24.9,N,2,32798.0,Busan
P0102,48.0,F,control,161.7,157.3,23.2,Y,3,47229.0,Busan
P0103,63.0,F,treatment,167.4,163.1,28.6,N,2,9597.0,Busan
P0104,56.0,F,treatment,136.4,117.3,27.9,N,4,27215.0,Daegu
P0105,62.0,F,control,144.3,141.1,26.9,N,5,40688.0,Daegu
P0106,41.0,F,treatment,162.2,151.1,30.6,N,4,25559.0,Seoul
P0107,54.0,F,treatment,144.2,128.9,26.7,N,4,37429.0,Daegu
P0108,28.0,F,control,148.0,143.9,23.8,N,5,40623.0,Daegu
P0109,59.0,F,control,147.8,147.4,23.4,N,1,25702.0,Daegu
P0110,54.0,F,control,168.9,175.6,26.7,Y,2,78180.0,Seoul
P0111,49.0,F,control,175.1,170.3,29.2,N,5,57014.0,Seoul
P0112,71.0,M,control,139.5,143.8,19.4,Y,4,13156.0,Gwangju
P0113,56.0,M,treatment,170.8,148.1,30.2,N,4,12769.0,Gwangju
P0114,66.0,F,control,130.1,130.0,24.4,N,1,41639.0,Gwangju
P0115,59.0,F,control,144.4,144.5,27.8,N,1,31583.0,Daegu
P0116,49.0,M,control,151.1,160.5,32.5,Y,2,95215.0,Gwangju
P0117,57.0,M,treatment,145.5,130.9,26.1,Y,1,25382.0,Daegu
P0118,52.0,F,control,137.3,,,N,4,23605.0,Daegu
P0119,31.0,F,control,132.7,141.8,24.8,N,2,6766.0,Seoul
P0120,64.0,F,treatment,119.6,101.6,29.3,N,5,22762.0,Daegu
P0121,72.0,M,control,165.0,156.9,23.2,N,1,12619.0,Gwangju
P0122,75.0,M,control,173.9,162.4,22.9,Y,4,11479.0,Seoul
P0123,49.0,M,control,142.6,135.5,24.1,Y,3,74950.0,Busan
P0124,66.0,F,treatment,167.8,154.9,19.7,N,3,49309.0,Daegu
P0125,37.0,F,control,168.1,163.2,27.1,N,2,37980.0,Busan
P0126,68.0,F,control,167.5,170.8,24.0,N,5,14773.0,Busan
P0127,55.0,F,treatment,119.6,,23.5,N,3,38653.0,Gwangju
P0128,70.0,F,control,153.7,147.5,25.6,N,5,15347.0,Daegu
P0129,63.0,F,treatment,175.8,154.4,22.3,N,1,21481.0,Gwangju
P0130,45.0,F,control,145.8,146.8,25.7,N,3,65265.0,Busan
P0131,51.0,M,treatment,157.3,150.8,25.0,N,5,11772.0,Daegu
P0132,49.0,F,treatment,146.1,136.6,30.7,N,4,35350.0,Daegu
P0133,61.0,M,control,161.7,157.6,23.8,Y,4,39063.0,Daegu
P0134,68.0,M,treatment,124.9,9.0,27.2,N,5,31211.0,Daegu
P0135,35.0,F,treatment,121.8,121.1,26.0,Y,2,26634.0,Daegu
P0136,46.0,M,treatment,146.1,128.2,29.0,N,2,18184.0,Seoul
P0137,59.0,F,treatment,164.9,170.6,22.3,N,2,14512.0,Seoul
P0138,47.0,M,treatment,146.2,145.5,22.6,N,3,38313.0,Gwangju
P0139,52.0,M,treatment,128.1,114.8,26.9,N,5,153003.0,Daegu
P0140,53.0,M,control,159.3,150.5,23.7,N,3,31681.0,Seoul
P0141,56.0,M,control,152.8,150.4,26.1,Y,1,11898.0,Busan
P0142,60.0,M,control,134.9,139.9,28.5,Y,3,14893.0,Gwangju
P0143,53.0,F,treatment,154.9,133.3,28.1,Y,2,26853.0,Seoul
P0144,59.0,M,treatment,173.3,161.8,22.0,N,1,26175.0,Daegu
P0145,40.0,M,treatment,128.9,,26.3,N,4,18401.0,Daegu
P0146,49.0,M,control,121.1,109.8,23.3,N,3,29908.0,Seoul
P0147,48.0,M,treatment,132.0,117.1,22.8,N,2,21847.0,Daegu
P0148,65.0,M,treatment,137.5,,27.4,Y,3,40174.0,Gwangju
P0149,51.0,F,treatment,154.0,152.7,21.5,N,2,20761.0,Daegu
P0150,61.0,F,control,156.5,147.6,19.7,Y,4,30679.0,Busan
P0151,29.0,M,control,134.7,129.5,29.5,Y,5,10778.0,Daegu
P0152,38.0,M,control,130.7,124.8,26.4,N,1,39255.0,Daegu
P0153,36.0,F,treatment,121.2,111.4,21.9,N,2,70491.0,Seoul
P0154,61.0,M,control,169.9,167.4,20.4,N,3,31538.0,Daegu
P0155,45.0,F,treatment,177.4,170.8,26.4,N,1,35181.0,Seoul
P0156,59.0,M,treatment,146.7,127.6,16.0,N,4,19414.0,Seoul
P0157,44.0,M,control,161.3,,,N,5,50047.0,Daegu
P0158,56.0,M,control,151.4,151.0,27.7,N,4,77464.0,Daegu
P0159,49.0,F,treatment,148.0,132.3,22.1,Y,3,28447.0,Seoul
P0160,38.0,F,control,126.6,125.9,26.6,N,1,25910.0,Seoul
P0161,63.0,F,treatment,137.1,116.3,32.5,N,1,59235.0,Seoul
P0162,57.0,F,treatment,141.5,126.4,29.3,N,5,25987.0,Gwangju
P0163,58.0,M,treatment,167.1,156.4,30.1,N,1,17583.0,Daegu
P0164,59.0,F,treatment,153.1,140.0,16.0,N,1,24298.0,Daegu
P0165,48.0,M,treatment,137.0,121.3,25.6,N,5,17519.0,Busan
P0166,55.0,F,control,168.1,161.8,24.3,N,1,43619.0,Busan
P0167,45.0,F,treatment,135.0,118.8,25.8,Y,2,22110.0,Daegu
P0168,60.0,M,treatment,147.4,135.6,29.3,N,4,30704.0,Daegu
P0169,37.0,F,control,135.1,128.0,32.3,N,4,16009.0,Seoul
P0170,76.0,M,control,172.8,175.9,26.5,N,3,23731.0,Daegu
P0171,60.0,M,treatment,153.8,136.3,23.7,N,4,18904.0,Daegu
P0172,58.0,F,control,130.9,125.2,22.3,Y,1,14560.0,Seoul
P0173,43.0,M,treatment,142.7,128.0,20.9,N,1,19676.0,Gwangju
P0174,74.0,F,treatment,138.1,130.7,24.6,N,1,22869.0,Seoul
P0175,61.0,F,control,145.6,129.1,22.7,Y,2,42134.0,Seoul
P0176,65.0,M,treatment,126.7,118.6,20.2,N,1,129301.0,Busan
P0177,62.0,M,treatment,150.3,136.0,27.8,N,3,55536.0,Daegu
P0178,62.0,F,treatment,146.7,134.0,20.3,N,2,24833.0,Daegu
P0179,70.0,F,control,153.7,141.8,30.0,N,1,20660.0,Seoul
P0180,54.0,F,treatment,143.6,130.4,29.4,Y,2,22577.0,Busan


In [ ]:
%%writefile sample_scores.csv
student_id,method,study_hours,score
S001,lecture,6.6,75.1
S002,lecture,4.3,68.0
S003,lecture,7.2,79.7
S004,lecture,6.9,68.0
S005,lecture,8.7,69.9
S006,lecture,4.3,71.7
S007,lecture,2.0,65.3
S008,lecture,8.9,80.2
S009,lecture,8.9,61.7
S010,lecture,3.4,74.8
S011,lecture,9.3,71.1
S012,lecture,8.0,79.7
S013,lecture,3.2,81.3
S014,lecture,4.7,81.6
S015,lecture,5.9,76.9
S016,lecture,6.8,66.1
S017,lecture,5.9,75.1
S018,lecture,2.9,75.7
S019,lecture,7.7,66.1
S020,lecture,5.8,66.1
S021,lecture,6.7,75.0
S022,lecture,7.5,64.4
S023,lecture,7.1,70.0
S024,lecture,4.9,69.9
S025,lecture,6.6,70.5
S026,lecture,10.3,70.5
S027,lecture,6.0,69.6
S028,lecture,13.9,94.4
S029,lecture,6.1,59.9
S030,lecture,10.0,68.4
S031,flipped,7.5,70.1
S032,flipped,6.2,84.3
S033,flipped,8.8,93.9
S034,flipped,6.1,70.6
S035,flipped,8.8,73.6
S036,flipped,7.2,72.3
S037,flipped,6.0,92.1
S038,flipped,4.8,61.0
S039,flipped,4.1,79.4
S040,flipped,4.8,76.4
S041,flipped,5.2,78.5
S042,flipped,5.9,70.7
S043,flipped,7.0,79.9
S044,flipped,7.6,84.3
S045,flipped,2.1,70.3
S046,flipped,4.3,73.8
S047,flipped,6.6,79.9
S048,flipped,10.1,76.1
S049,flipped,1.8,81.5
S050,flipped,5.6,68.3
S051,flipped,4.6,95.8
S052,flipped,5.3,85.7
S053,flipped,5.9,78.0
S054,flipped,5.8,84.8
S055,flipped,5.7,76.1
S056,flipped,6.0,75.4
S057,flipped,4.4,65.6
S058,flipped,4.2,68.6
S059,flipped,5.3,79.3
S060,flipped,1.8,73.0
S061,project,7.4,75.3
S062,project,8.1,82.5
S063,project,5.7,70.4
S064,project,7.2,64.2
S065,project,6.0,84.8
S066,project,2.2,90.6
S067,project,5.6,89.6
S068,project,5.5,88.3
S069,project,4.8,74.2
S070,project,5.9,73.7
S071,project,7.6,80.9
S072,project,8.3,71.9
S073,project,4.9,80.9
S074,project,8.5,76.8
S075,project,6.0,75.7
S076,project,4.7,78.7
S077,project,6.0,63.9
S078,project,7.9,72.0
S079,project,7.5,70.8
S080,project,8.0,84.3
S081,project,7.3,55.7
S082,project,3.9,84.3
S083,project,6.7,81.7
S084,project,6.3,71.5
S085,project,5.7,76.4
S086,project,5.0,81.3
S087,project,8.4,72.0
S088,project,4.1,80.1
S089,project,4.5,88.3
S090,project,7.9,80.3


In [ ]:
%%writefile sample_survey.csv
respondent_id,smoking,disease,age_group,exercise
R001,N,N,20s,sometimes
R002,N,N,30s,sometimes
R003,Y,N,20s,never
R004,Y,N,20s,often
R005,N,N,30s,sometimes
R006,Y,Y,40s,often
R007,Y,N,20s,sometimes
R008,Y,N,20s,never
R009,Y,N,20s,sometimes
R010,Y,Y,30s,often
R011,N,N,20s,sometimes
R012,N,N,30s,never
R013,Y,N,20s,often
R014,N,Y,20s,sometimes
R015,N,N,30s,never
R016,Y,Y,40s,sometimes
R017,N,Y,30s,often
R018,N,N,50s+,sometimes
R019,N,N,20s,often
R020,N,Y,30s,sometimes
R021,N,N,50s+,never
R022,N,N,40s,often
R023,N,N,40s,never
R024,N,Y,50s+,never
R025,Y,Y,20s,never
R026,N,N,20s,sometimes
R027,N,N,50s+,often
R028,N,N,20s,sometimes
R029,Y,Y,20s,often
R030,N,N,50s+,sometimes
R031,N,N,20s,never
R032,N,N,50s+,never
R033,N,N,30s,sometimes
R034,N,N,50s+,sometimes
R035,N,N,40s,sometimes
R036,N,N,20s,sometimes
R037,N,N,50s+,often
R038,Y,Y,50s+,sometimes
R039,N,N,40s,sometimes
R040,N,N,20s,never
R041,Y,N,50s+,sometimes
R042,N,N,50s+,sometimes
R043,N,N,20s,often
R044,N,N,40s,sometimes
R045,N,N,30s,sometimes
R046,N,N,30s,often
R047,N,N,20s,sometimes
R048,N,Y,20s,never
R049,Y,Y,50s+,never
R050,N,N,30s,sometimes
R051,Y,N,20s,sometimes
R052,N,N,20s,sometimes
R053,N,N,40s,sometimes
R054,N,N,50s+,often
R055,N,N,20s,often
R056,N,N,30s,sometimes
R057,Y,Y,50s+,never
R058,N,N,40s,never
R059,N,N,40s,sometimes
R060,N,N,30s,never
R061,Y,N,20s,never
R062,N,N,20s,sometimes
R063,N,N,20s,never
R064,N,N,40s,often
R065,N,N,20s,never
R066,N,N,50s+,often
R067,N,Y,40s,sometimes
R068,N,N,50s+,often
R069,Y,N,40s,never
R070,Y,N,20s,sometimes
R071,N,N,30s,never
R072,N,Y,20s,often
R073,N,Y,20s,often
R074,N,N,30s,never
R075,Y,N,30s,sometimes
R076,Y,N,40s,often
R077,Y,Y,50s+,sometimes
R078,N,N,20s,sometimes
R079,N,N,40s,never
R080,N,N,40s,never
R081,N,N,30s,sometimes
R082,N,N,20s,never
R083,N,N,40s,never
R084,N,N,50s+,sometimes
R085,N,N,40s,sometimes
R086,N,N,50s+,sometimes
R087,N,N,50s+,often
R088,N,N,40s,never
R089,Y,N,30s,sometimes
R090,N,N,50s+,never
R091,N,N,40s,never
R092,N,Y,20s,never
R093,Y,Y,20s,often
R094,N,N,20s,often
R095,N,N,40s,sometimes
R096,N,N,30s,sometimes
R097,N,N,30s,sometimes
R098,N,N,50s+,sometimes
R099,N,N,50s+,sometimes
R100,N,N,30s,often
R101,N,N,50s+,often
R102,N,N,40s,sometimes
R103,N,N,20s,often
R104,Y,N,20s,often
R105,N,N,40s,never
R106,N,Y,50s+,never
R107,N,N,20s,sometimes
R108,N,N,20s,sometimes
R109,N,N,30s,sometimes
R110,N,Y,30s,never
R111,Y,Y,30s,often
R112,N,N,50s+,sometimes
R113,N,N,40s,sometimes
R114,Y,N,30s,often
R115,N,N,50s+,never
R116,N,N,40s,sometimes
R117,N,N,30s,often
R118,N,N,40s,never
R119,N,N,20s,often
R120,Y,Y,20s,never
R121,Y,N,50s+,often
R122,N,N,50s+,never
R123,N,N,40s,sometimes
R124,N,N,50s+,sometimes
R125,N,N,30s,sometimes
R126,N,N,30s,sometimes
R127,N,N,30s,never
R128,N,N,40s,never
R129,N,Y,40s,sometimes
R130,Y,Y,40s,never
R131,N,Y,40s,sometimes
R132,N,N,20s,never
R133,N,N,50s+,never
R134,N,N,20s,sometimes
R135,Y,N,40s,sometimes
R136,Y,N,50s+,sometimes
R137,Y,Y,50s+,sometimes
R138,N,N,50s+,sometimes
R139,N,N,50s+,never
R140,N,N,30s,sometimes
R141,Y,N,30s,often
R142,N,Y,40s,sometimes
R143,Y,N,30s,never
R144,N,Y,20s,never
R145,N,N,40s,often
R146,N,N,20s,sometimes
R147,N,Y,30s,sometimes
R148,N,N,20s,sometimes
R149,N,N,30s,often
R150,Y,N,20s,sometimes
R151,Y,Y,20s,never
R152,N,N,20s,sometimes
R153,N,N,30s,sometimes
R154,N,Y,30s,sometimes
R155,Y,N,30s,sometimes
R156,Y,N,50s+,sometimes
R157,N,N,30s,often
R158,N,N,50s+,sometimes
R159,Y,N,50s+,sometimes
R160,N,N,30s,sometimes
R161,N,N,40s,sometimes
R162,N,N,20s,never
R163,Y,N,50s+,never
R164,N,N,50s+,sometimes
R165,N,N,20s,never
R166,N,N,40s,sometimes
R167,N,N,30s,sometimes
R168,N,N,30s,sometimes
R169,Y,N,50s+,sometimes
R170,N,N,20s,often
R171,N,Y,30s,never
R172,N,N,30s,often
R173,N,N,50s+,sometimes
R174,N,N,50s+,sometimes
R175,N,N,30s,often
R176,N,Y,30s,never
R177,N,N,50s+,never
R178,Y,N,40s,never
R179,N,N,30s,sometimes
R180,Y,Y,40s,never
R181,N,N,30s,sometimes
R182,N,N,20s,never
R183,Y,Y,50s+,sometimes
R184,N,N,40s,sometimes
R185,Y,Y,50s+,never
R186,N,Y,50s+,sometimes
R187,Y,Y,20s,never
R188,Y,N,50s+,sometimes
R189,N,N,30s,never
R190,Y,Y,20s,often
R191,N,N,30s,never
R192,Y,Y,50s+,never
R193,N,N,50s+,never
R194,Y,Y,30s,sometimes
R195,N,N,50s+,never
R196,Y,Y,40s,often
R197,Y,Y,20s,never
R198,N,N,50s+,sometimes
R199,N,N,30s,often
R200,N,N,30s,sometimes
R201,N,N,30s,sometimes
R202,N,N,40s,sometimes
R203,Y,Y,50s+,often
R204,N,N,20s,sometimes
R205,N,N,30s,sometimes
R206,Y,N,40s,often
R207,N,N,50s+,sometimes
R208,N,N,20s,sometimes
R209,N,N,20s,never
R210,Y,Y,50s+,sometimes
R211,N,N,50s+,often
R212,N,N,30s,often
R213,N,Y,30s,sometimes
R214,N,N,40s,often
R215,N,N,20s,never
R216,N,N,30s,often
R217,N,N,50s+,never
R218,Y,N,30s,often
R219,N,Y,20s,often
R220,N,N,20s,sometimes
R221,N,N,40s,never
R222,N,Y,30s,never
R223,N,N,40s,often
R224,Y,N,20s,never
R225,N,N,30s,sometimes
R226,N,N,40s,sometimes
R227,N,N,30s,often
R228,Y,N,30s,sometimes
R229,Y,N,40s,sometimes
R230,N,N,50s+,sometimes
R231,N,N,20s,often
R232,N,N,40s,often
R233,N,N,30s,never
R234,N,N,30s,never
R235,N,N,40s,sometimes
R236,Y,Y,30s,never
R237,N,N,40s,sometimes
R238,N,N,50s+,sometimes
R239,Y,N,40s,sometimes
R240,N,N,40s,never


In [ ]:
# 실습용 데이터 내려받기

import pandas as pd
clinic = pd.read_csv("sample_clinic.csv")
print("임상 데이터:", clinic.shape)
clinic.head()

---
## 1. 나쁜 설계부터 보기

초보자가 가장 많이 하는 실수는 **만능 도구 하나**를 만드는 것입니다.

In [ ]:
# 나쁜 예 -- 하지 마세요
def analyze(data, question):
    """데이터와 질문을 받아 알아서 분석한다."""
    ...

print("이 설계의 문제:")
print("  1. 모델이 할 일이 없다 -- 질문을 그대로 넘기고 끝")
print("  2. 안에서 무슨 일이 일어나는지 아무도 모른다")
print("  3. 실행 경로가 기록되지 않는다")
print("  4. 재사용이 안 된다")

좋은 설계는 반대입니다.

```
describe_column(column)
check_normality(column, group_column?)
compare_two_groups(value_column, group_column, method?, alpha?)
compare_many_groups(...)
```

**작다 · 하나만 한다 · 결과가 구조적이다.**

그래야 모델이 실제로 **판단**을 하게 되고, 실행 경로가 기록되고, 재사용됩니다.

> 다만 너무 잘게 쪼개도 곤란합니다. 도구가 서른 개면 모델이 목록을 다 읽지
> 못하고 엉뚱한 것을 고릅니다. **8개 내외**가 소형 모델이 감당하는 규모입니다.

---
## 2. 직접 만들어 보기 (축약판)

In [ ]:
from dataclasses import dataclass
from typing import Any, Callable
import json

@dataclass
class 내_도구:
    name: str
    description: str
    parameters: dict
    func: Callable

    def 명세(self):
        args = []
        for p, spec in self.parameters.items():
            mark = "" if spec.get("required") else "?"
            enum = f"={'|'.join(spec['enum'])}" if "enum" in spec else ""
            args.append(f"{p}{mark}: {spec.get('type','str')}{enum}")
        return f"- {self.name}({', '.join(args)}) — {self.description}"

    def 검사(self, args):
        문제 = []
        for p, spec in self.parameters.items():
            if spec.get("required") and p not in args:
                문제.append(f"필수 인자 '{p}' 가 없습니다.")
        for given in args:
            if given not in self.parameters:
                문제.append(f"'{given}' 는 인자가 아닙니다. 가능: {list(self.parameters)}")
        for p, v in args.items():
            spec = self.parameters.get(p, {})
            if "enum" in spec and v not in spec["enum"]:
                문제.append(f"'{p}' 값 {v!r} 불가. 가능: {spec['enum']}")
        return 문제


평균도구 = 내_도구(
    name="column_mean",
    description="한 열의 평균을 구한다.",
    parameters={"column": {"type": "str", "desc": "수치형 열", "required": True}},
    func=lambda column: float(pd.to_numeric(clinic[column], errors="coerce").mean()),
)

print(평균도구.명세())
print()
print("검사(정상):", 평균도구.검사({"column": "bmi"}))
print("검사(누락):", 평균도구.검사({}))
print("검사(오타):", 평균도구.검사({"col": "bmi"}))

> **인자 검사는 3주차 `structured.validate()` 와 같은 발상**입니다.
> 모델이 준 것을 믿지 않고 코드가 확인합니다.

In [ ]:
class 내_레지스트리:
    def __init__(self):
        self.tools = {}
        self.log = []

    def 등록(self, tool):
        self.tools[tool.name] = tool

    def 목록(self):
        return "\n".join(t.명세() for t in self.tools.values())

    def 호출(self, name, args=None):
        """예외를 밖으로 던지지 않는다. 실패도 결과로 돌려준다."""
        args = args or {}
        if name not in self.tools:
            return {"ok": False,
                    "error": f"'{name}' 도구가 없습니다.",
                    "hint": f"가능한 도구: {list(self.tools)}"}
        t = self.tools[name]
        문제 = t.검사(args)
        if 문제:
            return {"ok": False, "error": "; ".join(문제), "hint": t.명세()}
        try:
            return {"ok": True, "value": t.func(**args)}
        except Exception as exc:
            return {"ok": False, "error": f"{type(exc).__name__}: {exc}",
                    "hint": "열 이름을 확인하세요."}


reg = 내_레지스트리()
reg.등록(평균도구)

print(reg.호출("column_mean", {"column": "bmi"}))
print(reg.호출("column_mean", {"column": "없는열"}))
print(reg.호출("없는도구"))

### 왜 `call()` 은 예외를 던지지 않는가

에이전트 루프 안에서 도구가 예외를 던지면 **루프 전체가 죽습니다.**
열 단계 중 세 번째에서 오타 하나 때문에 전부 날아갑니다.

대신 실패를 결과 객체로 감싸 돌려주면 모델이 읽고 고칠 수 있습니다.
3주차에서 `ask_json` 이 오류를 되먹였던 것과 **정확히 같은 발상**입니다.

---
## 3. 공식 구현 `tools.ToolRegistry`

축약판에 몇 가지가 더 있습니다.

| 추가 | 왜 |
|---|---|
| `@registry.tool(...)` 데코레이터 | 등록이 간결해진다 |
| 타입 검사 (`int`/`float`/`bool`) | 모델이 `"5"` 를 보낼 때 걸러낸다 |
| `ToolResult.observation()` | 모델에게 돌려줄 문자열로 바로 변환 |
| `registry.log` | 실행 이력 — 6주차 trace 의 씨앗 |

In [ ]:
from tools import Tool, ToolRegistry, ToolResult

reg2 = ToolRegistry()

@reg2.tool("row_count", "데이터의 행 수를 돌려준다.", {})
def _row_count():
    return {"n_rows": len(clinic)}

@reg2.tool("column_mean", "한 수치형 열의 평균을 구한다.",
           {"column": {"type": "str", "desc": "열 이름", "required": True},
            "digits": {"type": "int", "desc": "반올림 자릿수, 기본 4"}})
def _column_mean(column, digits=4):
    if column not in clinic.columns:
        raise KeyError(f"'{column}' 열이 없습니다. 있는 열: {list(clinic.columns)}")
    return round(float(pd.to_numeric(clinic[column], errors="coerce").mean()), digits)

print(reg2.describe())
print()
for name, args in [("row_count", {}),
                   ("column_mean", {"column": "bmi"}),
                   ("column_mean", {"column": "bmi", "digits": "둘"}),
                   ("column_mean", {"column": "없는열"})]:
    r = reg2.call(name, args)
    print(f"{name}({args})")
    print("   ", r.observation().replace(chr(10), " | ")[:150])

---
## 4. 좋은 오류 메시지 — 이번 주 가장 중요한 것

오류 메시지를 어떻게 쓰느냐가 **에이전트의 성능을 바꿉니다.**

| 나쁜 메시지 | 좋은 메시지 |
|---|---|
| `ValueError: invalid group` | `'region' 의 수준이 4개입니다 (['Busan',...]). 두 집단 비교에는 2개여야 합니다. 3개 이상이면 compare_many_groups 를 쓰세요.` |

좋은 오류 메시지에는 세 가지가 들어갑니다.
**무엇이 잘못됐는지 · 왜 잘못됐는지 · 무엇을 하면 되는지.**

마지막이 특히 중요합니다. 그 한 줄이 다음 시도를 성공으로 바꿉니다.

In [ ]:
from tools import StatToolbox

box = StatToolbox(clinic)
print("등록된 통계 도구", len(box), "개")
print()

# 모델이 흔히 하는 실수 4가지
실수들 = [
    ("compare_two_groups", {"value_column": "bmi", "group_column": "region"}),
    ("compare_many_groups", {"value_column": "bmi", "group_column": "group"}),
    ("describe_column", {"col": "bmi"}),
    ("correlate", {"column_a": "age", "column_b": "bmi", "method": "kendall"}),
]
for name, args in 실수들:
    r = box.call(name, args)
    print(f">>> {name}({args})")
    print(r.observation()[:230])
    print()

> 각 메시지가 **다음에 무엇을 하면 되는지** 말해 주는지 확인하세요.
> `compare_two_groups` 실패 메시지는 `compare_many_groups` 를 쓰라고 알려줍니다.
> 이 한 줄 덕분에 6주차 ReAct 루프가 스스로 회복할 수 있습니다.
>
> 여러분이 만든 도구의 오류 메시지는 사람이 읽는 로그가 아니라
> **모델에게 주는 지시**입니다. 그렇게 생각하고 쓰십시오.

---
## 연습문제

**문제 1.** `reg2` 에 `column_std` 도구를 추가하시오.
`ddof` 를 인자로 받고 0 또는 1만 허용하게 하시오.

**문제 2.** 위 실수 4가지에 대한 오류 메시지를 읽고,
각각 모델이 다음에 무엇을 할지 예측하시오. 실제로 그렇게 되는지는 6주차에 확인한다.

**문제 3.** 일부러 **나쁜 오류 메시지**를 내는 도구를 만들고
(`raise ValueError("error")`), 모델에게 고치게 해 보시오. 몇 번 만에 성공하는가?

**문제 4.** 도구 이름을 `f1`, `f2`, `f3` 처럼 무의미하게 바꾸면
모델의 선택 정확도가 떨어지는가? 다음 노트북의 측정 코드를 미리 써서 확인하시오.

In [ ]:
# 여기에 직접 작성해 보세요

<details>
<summary><b>해설 보기</b></summary>

**문제 1.**
```python
@reg2.tool("column_std", "한 수치형 열의 표준편차를 구한다.",
           {"column": {"type": "str", "required": True},
            "ddof":   {"type": "int", "enum": [0, 1], "desc": "1이면 표본표준편차"}})
def _column_std(column, ddof=1):
    return round(float(pd.to_numeric(clinic[column], errors="coerce").std(ddof=ddof)), 4)
```
`enum` 에 정수를 넣어도 동작합니다. 검증은 `in` 으로 하기 때문입니다.

**문제 4.** 크게 떨어집니다. 모델은 **이름과 설명을 읽고 판단**합니다.
`f1` 이라는 이름에서는 아무 정보도 얻을 수 없습니다.

이것이 시사하는 바가 있습니다. 도구 이름은 코드의 식별자가 아니라
**프롬프트의 일부**입니다. 사람이 읽기 좋은 이름이 모델에게도 좋습니다.

</details>

---
## 정리

- 도구는 **작고, 하나만 하고, 구조화된 결과**를 돌려준다
- 명세(`이름`·`설명`·`인자`)는 **모델이 읽을 문서**다
- 인자는 **코드가 검증한다.** 모델이 준 것을 믿지 않는다
- `call()` 은 **예외를 던지지 않는다.** 루프가 죽으면 안 된다
- 오류 메시지에 **무엇을 하면 되는지** 쓴다

**다음 노트북** → [`03_stat_tools.ipynb`](https://colab.research.google.com/github/zoomer1975-boop/ai_agent/blob/main/week05/03_stat_tools.ipynb)
모델에게 실제로 도구를 고르게 하고 정확도를 잽니다.